# Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier


LOADING THE DATASET

In [3]:
customer_transaction_data = pd.read_csv(r"C:\Users\pc\Downloads\Fraudulent_Transaction_Detection_Using_Machine_Learning\dataset\FinLora_Customer_Transaction_Dataset.csv")
customer_transaction_data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [4]:
# checking for number of rows and columns
customer_transaction_data.shape

(11400, 26)

In [5]:
# checking for all the columns in the dataset
customer_transaction_data.columns

Index(['transaction_id', 'customer_id', 'timestamp', 'home_country',
       'source_currency', 'dest_currency', 'channel', 'amount_src',
       'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id',
       'new_device', 'ip_address', 'ip_country', 'location_mismatch',
       'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud'],
      dtype='str')

checking of missing values in each of variables we can confirm that:
timestamp has 29 missing values

amount_usd has 305 missing values
fess has 295 missing values
ip_address has 305 missing values
ip_country has 301 missing values
kyc_tier has 300 missing values
device_trust_score has 295 missing values

In [6]:
# checking for missing data
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [7]:
customer_transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [8]:
# checking transactions numbers made
customer_transaction_data['amount_src'].value_counts(dropna=False).head(20)

amount_src
100.0     15
102.29     5
115.29     4
54.74      4
167.16     4
82.25      4
128.93     4
261.23     4
71.78      4
102.37     4
105.2      4
121.3      4
119.15     4
250.19     4
58.93      4
94.34      4
144.29     4
165.91     3
115.25     3
111.18     3
Name: count, dtype: int64

summary / observations

The composition of the dataset is 11400 rows and 26 columns
All fieldnames follows the standard variables naming  conventions
An initial inspection of the dataset revealed inconsistence in the data types of certain variables. specifically the timestamp variablesis currently stored as an object data type instead of the appropriate datetime format, which is neccessary for accurate time-based analysis, similarly the amount_src variable is also stored as an  object data type , whereas it should ideally be represented as a numeric format integer or floats to allow for proper numerical computations and analysis. These inconsistences will need to be corrected during the data proprocessing stage.

An assessement of missing values across the dataset revealed that several variables contain incomplete records, specifically the timestamp variables contains 29 missing values ip_country has 301 misssing values and kyc_tier contains 300 misssing values. These missing values will require appropriate data cleaning and preprocessing to ensure the reliability and accuracy of subsequent analysis and modelling.
The target variables is_fraud is highly imbalanced, with appropriate 91%  of transaction classified as  legitimate and only 9% as identified as fraudulent.
This imbalance introduce a risk of model bias towards the majority class where machine learning algorithms may prioritize predicting legitimate transaction while failing to detect fraudulent ones. As a result relying solely on accuracy could produce misleading results To mitigate this issues techniques such as resampling methods and appropriate evaluation metrics should be used to ensure the model effectively identifies  fraudulent transactions.

In [9]:
# checking for label imbalance
customer_transaction_data['is_fraud'].value_counts(normalize=True).reset_index()

,is_fraud,proportion
0,0,0.912544
1,1,0.087456


DATA CLEANING AND PREPARATION

In [10]:
# converting data types of timestamp to datetime
customer_transaction_data['timestamp'] = pd.to_datetime(customer_transaction_data['timestamp'],errors='coerce')

# converting data type of income_usd to float
customer_transaction_data['amount_src'] = pd.to_numeric(customer_transaction_data['amount_src'], errors='coerce')

In [11]:
# checking the dataset again to confrim the data type
customer_transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  str                
 1   customer_id                11400 non-null  str                
 2   timestamp                  11339 non-null  datetime64[us, UTC]
 3   home_country               11400 non-null  str                
 4   source_currency            11400 non-null  str                
 5   dest_currency              11400 non-null  str                
 6   channel                    11400 non-null  str                
 7   amount_src                 11396 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  device_id    

During data preprocessing the timestamp and amount_src variables were converted to appropriate data types using pd.to_datetime() and pd.to_numeric() with the errors="coerce" argument. This process revealed additional invalid entries that could not be parsed correctly, resulting in an increses in missing values. Specifically the number of missing values in timestamp inceased from 29 to 61, while amount_src increased from 0 to 4. These additional missing values indicate the presence of improperly formated or corrupted records which were subsequently handled during the data cleaning stage

In [12]:
# Rechecking for missing data
customer_transaction_data.isnull().sum()


transaction_id                 0
customer_id                    0
timestamp                     61
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     4
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

Nxt is to filling the missing values
VARIABLES WITH MISSING VALUES THAT NEEDS IMPUTATION

amount_usd : has 305 missing values this will be filled by multipling the amount_src by the exchange rate of the currency
fess: has 295 missing values which will be filled by the median of the channel variable if channel exist  while other remaining fee missing values will be filled using the overall median

ip_country has 301 missing values which will be filled by the corresponding home_country
kyc_tier has 300 mising values whic will be filled with the most frequent kyc_tier(mode)
device_trust_score has 295 missing values this will be filled by the median of the of the group from gouping device_trust_score by the kyc_tier and new_device variables or by the overall median of device_trust_score

The timestamp variable whic has 61 missing values and amount_src with 4 missing values will be dropped using the dropna method because the number of missing  values in both variable is very samll compared to the values we have on our dataset. The ip_address variables has missing values of 305 cannot be filled  with any value as each ip_address is a unique numberb so it will be dropped using the dropna method

In [13]:
# calculating exchange rates per currency 
exchange_rates = customer_transaction_data[customer_transaction_data['amount_usd'].notna()].groupby('source_currency').apply(
    lambda x: (x['amount_usd'] / x['amount_src']).mean()
).to_dict()
exchange_rates

{'CAD': 0.7216095926871465,
 'GBP': 1.223441221648679,
 'USD': 0.9838730321259439}

In [14]:
# filling the missing values of the amount_usd
customer_transaction_data['amount_usd']=customer_transaction_data.apply(
    lambda row: row['amount_usd'] if pd.notna(row['amount_usd']) else row['amount_src'] * exchange_rates.get(row['source_currency'],1),
    axis=1
)

Note the following in solving nxt 


In [15]:
customer_transaction_data[['channel', 'fee']]

,channel,fee
0,ATM,4.25
1,web,4.24
2,mobile,2.70
3,mobile,2.22
4,mobile,3.61
...,...,...
11395,mobile,5.24
11396,web,8.86
11397,web,4.00
11398,mobile,1.96


In [16]:
# filling the missing values of the fee variable
if 'fee' in customer_transaction_data.columns:
    if 'channel' in customer_transaction_data.columns:
        customer_transaction_data['fee'] = customer_transaction_data.groupby('channel')['fee'].transform(lambda y: y.fillna(y.median()))
    customer_transaction_data['fee'] = customer_transaction_data['fee'].median()

In [17]:
# filling the missing values of ip_country variable
if {'ip_country', 'home_county'}.issubset(customer_transaction_data.columns):
    customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].fillna(customer_transaction_data['home_country'])

In [18]:
# filling the missing values with device_trust_score variable
if 'device_trust_score' in customer_transaction_data.columns:
    if{'new_device', 'kyc_tier'}.issubset(customer_transaction_data.columns):
        customer_transaction_data['device_trust_score']=customer_transaction_data.groupby(['new_device', 'kyc_tier'])['device_trust_score'].transform(lambda x: x.fillna(x.median()))
    customer_transaction_data['device_trust_score']=customer_transaction_data['device_trust_score'].fillna(customer_transaction_data['device_trust_score'].median())

In [19]:
# dropping the timestamp, amount_src and ip_address missing values
customer_transaction_data.dropna(subset=['timestamp','amount_src', 'ip_address'], inplace=True)

C:\Users\pc\AppData\Local\Temp\ipykernel_18648\4178595318.py:2: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  customer_transaction_data.dropna(subset=['timestamp','amount_src', 'ip_address'], inplace=True)


In [20]:
# rechecking for missing values
customer_transaction_data.isnull().sum()

transaction_id               0
customer_id                  0
timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
device_id                    0
new_device                   0
ip_address                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
dtype: int64

STEPS TO TAKE TO DO SANITY CHECK
check for invalid numbers ie negative numbers for variables that contain age, monetary values, velocity  etc
check if transaction timestamp appear in the future
check that values falls btw range
inspect for inconsistency among variable values in the dataset

In [21]:
# checking negative values in numeric columns
negative_counts = {
    'amount_src': (customer_transaction_data['amount_src'] < 0).sum(),
    'amount_usd': (customer_transaction_data['amount_usd'] < 0).sum(),
    'fee': (customer_transaction_data['fee']<0).sum(),
    'device_trust_score': (customer_transaction_data['device_trust_score']<0).sum(),
    'txn_velocity_1h': (customer_transaction_data['txn_velocity_1h'] <0).sum(),
    'txn_velocity_24h': (customer_transaction_data['txn_velocity_24h'] <0).sum(),
    'risk_score_internal': (customer_transaction_data['risk_score_internal'] < 0).sum(),
}
negative_counts

{'amount_src': np.int64(100),
 'amount_usd': np.int64(0),
 'fee': np.int64(0),
 'device_trust_score': np.int64(190),
 'txn_velocity_1h': np.int64(190),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

Note that all the negative values will drop then. whic is discovered in above.

In [22]:
customer_transaction_data = customer_transaction_data.drop(customer_transaction_data[
    (customer_transaction_data['amount_src'] <0) |
    (customer_transaction_data['fee'] < 0) |
    (customer_transaction_data['device_trust_score'] < 0) |
    (customer_transaction_data['txn_velocity_1h'] <0)].index)

In [23]:
# checking for futuristic timestamp
customer_transaction_data[customer_transaction_data['timestamp'] > pd.Timestamp.utcnow()]

C:\Users\pc\AppData\Local\Temp\ipykernel_18648\1800480428.py:2: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  customer_transaction_data[customer_transaction_data['timestamp'] > pd.Timestamp.utcnow()]


,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud


In [24]:
# checking for location mismatch
customer_transaction_data['location_mismatch'].value_counts().reset_index()

,location_mismatch,count
0,False,9047
1,True,1793


In [25]:
# reviewing the channel variable
customer_transaction_data['channel'].unique()

<ArrowStringArray>
[      'ATM',       'web',    'mobile',       'WEB',    ' web  ',    'MOBILE',
   'unknown',   'mobille', ' mobile  ',      'weeb',       'ATm',    ' ATM  ']
Length: 12, dtype: str

In [26]:
# formatting the channel variable
customer_transaction_data['channel'] = customer_transaction_data['channel'].str.lower().str.strip()
customer_transaction_data['channel'].unique()

<ArrowStringArray>
['atm', 'web', 'mobile', 'unknown', 'mobille', 'weeb']
Length: 6, dtype: str

In [28]:
# we try to hamornize the mis-spelled
customer_transaction_data['channel'] = customer_transaction_data['channel'].replace({
    'weeb':'web',
    'mobille':'mobile',
})

In [29]:
customer_transaction_data['channel'] = customer_transaction_data['channel'].replace({'unknown':np.nan})
customer_transaction_data['channel'].unique()

<ArrowStringArray>
['atm', 'web', 'mobile', nan]
Length: 4, dtype: str

In [30]:
# reviewing source currency variable
customer_transaction_data['source_currency'].unique()

<ArrowStringArray>
['USD', 'CAD', 'GBP']
Length: 3, dtype: str

In [31]:
# reviewing the destination currency variable
customer_transaction_data['dest_currency'].unique()

<ArrowStringArray>
['CAD', 'MXN', 'CNY', 'EUR', 'INR', 'GBP', 'PHP', 'NGN', 'USD']
Length: 9, dtype: str

In [32]:
# review the kyc_tier variable
customer_transaction_data['kyc_tier'].unique()

<ArrowStringArray>
[   'standard',    'enhanced',         'low', ' standard  ',     'standrd',
 ' enhanced  ',    'STANDARD',     'unknown',     'enhancd',      ' low  ',
    'ENHANCED',         'LOW']
Length: 12, dtype: str

In [33]:
#formatting the channel variable
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].str.lower().str.strip()
customer_transaction_data['kyc_tier'].unique()

<ArrowStringArray>
['standard', 'enhanced', 'low', 'standrd', 'unknown', 'enhancd']
Length: 6, dtype: str

In [34]:
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].replace({
    'standrd':'standard',
    'enhancd':'enhanced',
    'unknown':'np.nan'
})
customer_transaction_data['kyc_tier'].unique()

<ArrowStringArray>
['standard', 'enhanced', 'low', 'np.nan']
Length: 4, dtype: str

In [35]:
customer_transaction_data.isnull().sum()

transaction_id                0
customer_id                   0
timestamp                     0
home_country                  0
source_currency               0
dest_currency                 0
channel                      36
amount_src                    0
amount_usd                    0
fee                           0
exchange_rate_src_to_dest     0
device_id                     0
new_device                    0
ip_address                    0
ip_country                    0
location_mismatch             0
ip_risk_score                 0
kyc_tier                      0
account_age_days              0
device_trust_score            0
chargeback_history_count      0
risk_score_internal           0
txn_velocity_1h               0
txn_velocity_24h              0
corridor_risk                 0
is_fraud                      0
dtype: int64

In [36]:
customer_transaction_data.dropna(inplace=True)
customer_transaction_data.info()

C:\Users\pc\AppData\Local\Temp\ipykernel_18648\1034264190.py:1: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  customer_transaction_data.dropna(inplace=True)


<class 'pandas.DataFrame'>
Index: 10804 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             10804 non-null  str                
 1   customer_id                10804 non-null  str                
 2   timestamp                  10804 non-null  datetime64[us, UTC]
 3   home_country               10804 non-null  str                
 4   source_currency            10804 non-null  str                
 5   dest_currency              10804 non-null  str                
 6   channel                    10804 non-null  str                
 7   amount_src                 10804 non-null  float64            
 8   amount_usd                 10804 non-null  float64            
 9   fee                        10804 non-null  float64            
 10  exchange_rate_src_to_dest  10804 non-null  float64            
 11  device_id         

In [37]:
# reviewing the home_country variable
customer_transaction_data['home_country'].unique()

<ArrowStringArray>
['US', 'CA', 'UK', ' UK  ', ' US  ', 'unknown', ' CA  ']
Length: 7, dtype: str

In [38]:
#formatting the home_country variable
customer_transaction_data['home_country'] = customer_transaction_data['home_country'].str.lower().str.strip()
customer_transaction_data['home_country'].unique()

<ArrowStringArray>
['us', 'ca', 'uk', 'unknown']
Length: 4, dtype: str

In [39]:
# reviewing the ip_country variable
customer_transaction_data['ip_country'].unique()


<ArrowStringArray>
['US', 'CA', 'UK', ' US  ', 'unknown', ' CA  ', ' UK  ']
Length: 7, dtype: str

In [40]:
# formatting the ip_country variables
customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].str.lower().str.strip()
customer_transaction_data['ip_country'].unique()

<ArrowStringArray>
['us', 'ca', 'uk', 'unknown']
Length: 4, dtype: str

In [41]:
customer_transaction_data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,us,USD,CAD,atm,278.19,278.19,3.48,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,ca,CAD,MXN,web,208.51,154.29,3.48,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,us,USD,CNY,mobile,160.33,160.33,3.48,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,us,USD,EUR,mobile,59.41,59.41,3.48,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,us,USD,INR,mobile,200.96,200.96,3.48,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


Solving the normalization problem for 
1. Home Country
2. ip country
3. kyc_tier

we now open a particular file os.makdirs is a command to open a new directory then we now store this dataset inside it.



In [51]:
import os
os.makedirs("../dataset/artifacts")

FileExistsError: [WinError 183] Cannot create a file when that file already exists: '../dataset/artifacts'

In [52]:
customer_transaction_data.to_csv(r'../dataset/artifacts/Cleaned_Data.csv',index=False)